# Juge corrigé — notebook autonome

**Ne dépend d'aucune session précédente.** Recharge tout depuis zéro : imports, Drive, données déjà
générées, fonctions nécessaires, puis relance uniquement le juge (Phi-3-mini) avec le prompt corrigé.


In [ ]:
!pip install -q bitsandbytes accelerate "transformers==4.51.3" pandas numpy

import os
import re
import gc
import json
import torch
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU requis. Activez-le via Runtime > Change runtime type > GPU.")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")

drive.mount('/content/drive')
BASE_DRIVE = '/content/drive/MyDrive/Master_Thesis_RAG_Indexes'  # ajustez si double niveau chez vous
RESULTS_DIR = f"{BASE_DRIVE}/Experiments_Results/model_comparison_v2_scientifique"
print(f"📁 {RESULTS_DIR}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ GPU : Tesla T4
Mounted at /content/drive
📁 /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Experiments_Results/model_comparison_v2_scientifique


In [ ]:
# Chargement des données déjà générées 
generations_path = f"{RESULTS_DIR}/generations_evaluated.csv"
if not os.path.exists(generations_path):
    generations_path = f"{RESULTS_DIR}/generations_full.csv"

df_results = pd.read_csv(generations_path)
df_results["answer"] = df_results["answer"].fillna("").astype(str)
print(f"✅ {len(df_results)} générations chargées depuis {generations_path}")
print(f"   Modèles présents : {df_results['model'].unique().tolist()}")

✅ 225 générations chargées depuis /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Experiments_Results/model_comparison_v2_scientifique/generations_evaluated.csv
   Modèles présents : ['Qwen3-8B', 'Llama-3.1-8B', 'OpenBioLLM-8B']


In [ ]:
def load_model_4bit(hf_id, trust_remote_code=True):
    """Charge un modèle en 4 bits. trust_remote_code=False pour Phi-3 (le code distant du
    dépôt HF est incompatible avec les versions récentes de transformers ; Phi-3 est de
    toute façon nativement supporté, donc ce code personnalisé n'est pas nécessaire)."""
    gc.collect()
    torch.cuda.empty_cache()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_remote_code)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, quantization_config=bnb_config, device_map={"": 0}, trust_remote_code=trust_remote_code
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def render_prompt(tokenizer, prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt_text + "\n\nRéponse :"

print("✅ Fonctions prêtes")

✅ Fonctions prêtes


## Juge corrigé

**Corrections vs la version initiale** : plus de champ `justification` demandé (source de troncature
du JSON), prompt réordonné pour finir sur les scores, et repli par extraction regex champ par champ
si le JSON global ne se parse pas — la version initiale n'obtenait que 14 à 29 évaluations valides
sur 75 par modèle, cette version vise un taux de complétude nettement supérieur.

In [ ]:
JUDGE_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
judge_model, judge_tokenizer = load_model_4bit(JUDGE_MODEL_ID, trust_remote_code=False)
print(f"✅ Juge chargé : {JUDGE_MODEL_ID}")

JUDGE_PROMPT = """Vous êtes un évaluateur clinique expert. Évaluez la réponse suivante sur 7 dimensions,
chacune notée de 1 (très faible) à 5 (excellent). Répondez UNIQUEMENT au format JSON, sans texte avant ni après.

Question : {question}
Contexte fourni au modèle : {context}
Réponse à évaluer : {answer}

Format strict, SANS justification (juste les 7 notes) :
{{"clinical_accuracy": X, "groundedness": X, "safety": X, "completeness": X, "reasoning": X, "language_quality": X, "citation_quality": X}}
"""

FIELDS = ["clinical_accuracy", "groundedness", "safety", "completeness", "reasoning", "language_quality", "citation_quality"]

def judge_answer(row):
    contexts = json.loads(row["contexts"])
    prompt = JUDGE_PROMPT.format(
        question=row["question"], context="\n".join(contexts)[:2000], answer=row["answer"]
    )
    rendered = render_prompt(judge_tokenizer, prompt)
    inputs = judge_tokenizer(rendered, return_tensors="pt", truncation=True, max_length=3000).to(judge_model.device)
    with torch.no_grad():
        output = judge_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    raw_text = judge_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    json_match = re.search(r"\{.*?\}", raw_text, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group(0))
            if all(f in parsed for f in FIELDS):
                return parsed
        except Exception:
            pass

    result = {}
    for field in FIELDS:
        m = re.search(rf'"{field}"\s*:\s*(\d)', raw_text)
        if m:
            result[field] = int(m.group(1))
    return result if result else None

judge_scores = []
n_full_success = 0
for i, row in df_results.iterrows():
    result = judge_answer(row)
    if result and len(result) == 7:
        n_full_success += 1
    judge_scores.append({"model": row["model"], "id": row["id"], **(result if result else {})})
    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(df_results)}] — {n_full_success} évaluations complètes jusqu'ici")

judge_df_v2 = pd.DataFrame(judge_scores)
judge_df_v2.to_csv(f"{RESULTS_DIR}/judge_scores_corrected.csv", index=False)

del judge_model, judge_tokenizer
gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Juge corrigé sauvegardé séparément : judge_scores_corrected.csv")
print(f"   {n_full_success}/{len(df_results)} évaluations complètes")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Juge chargé : microsoft/Phi-3-mini-4k-instruct
  [25/225] — 21 évaluations complètes jusqu'ici
  [50/225] — 45 évaluations complètes jusqu'ici
  [75/225] — 54 évaluations complètes jusqu'ici
  [100/225] — 74 évaluations complètes jusqu'ici
  [125/225] — 96 évaluations complètes jusqu'ici
  [150/225] — 112 évaluations complètes jusqu'ici
  [175/225] — 134 évaluations complètes jusqu'ici
  [200/225] — 157 évaluations complètes jusqu'ici
  [225/225] — 180 évaluations complètes jusqu'ici

✅ Juge corrigé sauvegardé séparément : judge_scores_corrected.csv
   180/225 évaluations complètes
